In [1]:
import litellm
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

print(f"Libraries and environment variables loaded sucessfuly")

Libraries and environment variables loaded sucessfuly


In [36]:
MODEL_PRICING = {
    "gpt-4o-mini": {
        "input_per_1m": 0.15,
        "output_per_1m": 0.60
    }
}

MODEL_NAME = "openai/gpt-4o-mini"

price_info = MODEL_PRICING[MODEL_NAME.split("/")[-1]]
PRICE_PER_INPUT_TOKEN = price_info["input_per_1m"] / 1_000_000
PRICE_PER_OUTPUT_TOKEN = price_info["output_per_1m"] / 1_000_000

print(f"pricing for model: {MODEL_NAME}")
print(f"pricing per input token: \t${PRICE_PER_INPUT_TOKEN:.10f}")
print(f"pricing per outpu token: \t${PRICE_PER_OUTPUT_TOKEN:.10f}")

pricing for model: openai/gpt-4o-mini
pricing per input token: 	$0.0000001500
pricing per outpu token: 	$0.0000006000


In [42]:
def get_completion(prompt, model, max_tokens=20):
    print("--- Getting completion from LiteLLM---")

    response = litellm.completion(
        model=model,
        messages=[
            {
                "role":"user",
                "content":"You are a helpful travel assistant"
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        max_tokens=max_tokens
    )

    return response

def analyze_cost(response):
    if not response:
        print("cannont analyze cost of a failed API")
        return
        
    raw_model_name = response.model

    if "/" in raw_model_name:
        raw_model_name = raw_model_name.split("/")[-1]
        
    model_used = None

    for model in MODEL_PRICING.keys():
        if raw_model_name.startswith(model):
            model_used = model
            break
    price_info = MODEL_PRICING[model_used]

    if not price_info:
        print(f"WARNING: Pricing for model '{model_used} not found")
        return
    usage_data = response.usage
    input_tokens = usage_data.prompt_tokens
    output_tokens = usage_data.completion_tokens

    price_per_input = price_info["input_per_1m"] / 1_000_000
    price_per_output = price_info["output_per_1m"] / 1_000_000

    input_cost = input_tokens * price_per_input
    output_cost = output_tokens * price_per_output
    total_cost = input_cost + output_cost

    print(f"\n-----------Cost Break-down for model: {model_used}-----")
    print(f"Input Tokens: \t{input_tokens} Cost: {input_cost:.8f}")
    print(f"output Tokens: \t{output_tokens} Cost: {output_cost:.8f}")
    print(f"Total Cost of API : \t{total_cost:.8f}")
        

user_prompt = "what is the capital of France, and what is it famous for?"


response = get_completion(
            user_prompt,
            model=MODEL_NAME
        )

      
        
        
    

--- Getting completion from LiteLLM---


In [43]:
analyze_cost(response)


-----------Cost Break-down for model: gpt-4o-mini-----
Input Tokens: 	31 Cost: 0.00000465
output Tokens: 	20 Cost: 0.00001200
Total Cost of API : 	0.00001665
